# OCR Plat Nomor dari Nol — CNN + BiLSTM + CTC

Notebook ini dibuat untuk **belajar fundamental OCR**, jadi tidak memakai EasyOCR/Tesseract.

Dataset Roboflow yang dipakai adalah **Plate-Recognition**:
https://universe.roboflow.com/muhammad-faris/plate-recognition-qzqlz-nuszg

Dataset tersebut berupa **object detection per karakter**. Kita akan memanfaatkan bounding box karakter hanya untuk:
1. meng-crop area teks plat saat menyiapkan data training, dan
2. menyusun ground-truth text dari kiri ke kanan.

Saat inference, model OCR **tidak membutuhkan bounding box karakter** lagi.

Pipeline yang dipelajari:

`plate image -> CNN -> feature sequence -> BiLSTM -> character logits -> CTC -> text`


## 1. Install dependency

Format download yang dipakai: **YOLOv8**.

Alasannya sederhana: setiap anotasi karakter disimpan sebagai `.txt` berisi `class_id x_center y_center width height`, sehingga mudah kita urutkan berdasarkan posisi `x_center` untuk membuat label teks.

In [ ]:
!pip -q install roboflow pyyaml opencv-python-headless matplotlib

## 2. Download dataset dari Roboflow

API key dibaca secara interaktif agar tidak tersimpan di notebook.

> Jika Roboflow menampilkan version selain `1`, ubah nilai `VERSION` sesuai snippet **Download Dataset** di Roboflow.

In [ ]:
import os
from getpass import getpass
from roboflow import Roboflow

API_KEY = os.getenv("ROBOFLOW_API_KEY") or getpass("Roboflow API key: ")
if not API_KEY.strip():
    raise ValueError("ROBOFLOW_API_KEY/API key tidak boleh kosong")

WORKSPACE = "muhammad-faris"
PROJECT = "plate-recognition-qzqlz-nuszg"
VERSION = 1

rf = Roboflow(api_key=API_KEY)
project = rf.workspace(WORKSPACE).project(PROJECT)
dataset = project.version(VERSION).download("yolov8")

DATASET_DIR = dataset.location
print("Dataset:", DATASET_DIR)


## 3. Baca nama class

Class pada dataset adalah karakter seperti `0-9`, `A-Z`, dan class tambahan yang memang terdapat pada anotasi.

Kita membuat `blank = 0` khusus untuk CTC. Karena itu class dataset digeser `+1` saat menjadi target OCR.

In [ ]:
from pathlib import Path
import yaml

DATASET_DIR = Path(DATASET_DIR)
DATA_YAML = DATASET_DIR / "data.yaml"
if not DATA_YAML.is_file():
    raise FileNotFoundError(f"data.yaml tidak ditemukan: {DATA_YAML}")

with DATA_YAML.open("r") as f:
    config = yaml.safe_load(f)

class_names = config["names"]

# Roboflow bisa menyimpan names sebagai list atau dict.
if isinstance(class_names, dict):
    class_names = [
        class_names[k]
        for k in sorted(class_names, key=lambda x: int(x))
    ]

class_names = [str(name) for name in class_names]
if not class_names:
    raise ValueError("data.yaml tidak memiliki class OCR")

print("Jumlah class:", len(class_names))
print("Class:", class_names)

BLANK_ID = 0
NUM_CLASSES = len(class_names) + 1  # + blank untuk CTC

print("Jumlah output OCR:", NUM_CLASSES)


## 4. Ubah anotasi object detection menjadi pasangan `image -> text`

Contoh anotasi YOLO:

`12 0.20 0.50 0.08 0.60`

berarti karakter dengan `class_id=12` berada di suatu posisi pada gambar.

Untuk membuat ground truth OCR:
- baca semua karakter,
- kelompokkan baris berdasarkan posisi `y_center`, lalu urutkan kiri ke kanan,
- gabungkan nama class menjadi satu string.

Contoh:

`[B] [1] [2] [3] [4] [XYZ] -> B1234XYZ`

Urutan ini juga aman untuk plat dua baris sederhana: baris atas dibaca lebih dulu.

In [ ]:
import cv2
import numpy as np
import torch
from torch.utils.data import DataLoader, Dataset

IMG_W = 128
IMG_H = 32
CTC_TIME_STEPS = IMG_W // 4


def find_split_dir(root, split):
    candidates = {
        "train": ["train"],
        "valid": ["valid", "val"],
        "test": ["test"]
    }

    for name in candidates[split]:
        path = root / name
        if path.is_dir():
            return path

    return None


def sort_characters(chars):
    if len(chars) <= 1:
        return chars

    median_height = float(np.median([char["h"] for char in chars]))
    row_tolerance = max(median_height * 0.6, 0.02)
    rows = []

    for char in sorted(chars, key=lambda item: item["y"]):
        matches = [
            (abs(char["y"] - row["center_y"]), row)
            for row in rows
        ]

        if matches:
            distance, row = min(matches, key=lambda item: item[0])
        else:
            distance, row = float("inf"), None

        if row is not None and distance <= row_tolerance:
            row["chars"].append(char)
            row["center_y"] = np.mean([item["y"] for item in row["chars"]])
        else:
            rows.append({"center_y": char["y"], "chars": [char]})

    rows.sort(key=lambda row: row["center_y"])
    return [
        char
        for row in rows
        for char in sorted(row["chars"], key=lambda item: item["x"])
    ]


def read_yolo_characters(label_path):
    chars = []

    with open(label_path, "r") as f:
        for line_number, line in enumerate(f, start=1):
            if not line.strip():
                continue

            parts = line.strip().split()
            if len(parts) != 5:
                raise ValueError(f"Format label tidak valid di {label_path}:{line_number}")

            class_id = int(float(parts[0]))
            x_center, y_center, width, height = map(float, parts[1:])

            if not 0 <= class_id < len(class_names):
                raise ValueError(f"class_id {class_id} di luar range: {label_path}:{line_number}")
            if not all(np.isfinite(value) for value in (x_center, y_center, width, height)):
                raise ValueError(f"Koordinat tidak valid: {label_path}:{line_number}")
            if not 0 <= x_center <= 1 or not 0 <= y_center <= 1:
                raise ValueError(f"Center bbox di luar range 0..1: {label_path}:{line_number}")
            if not 0 < width <= 1 or not 0 < height <= 1:
                raise ValueError(f"Ukuran bbox tidak valid: {label_path}:{line_number}")

            chars.append({
                "class_id": class_id,
                "x": x_center,
                "y": y_center,
                "w": width,
                "h": height,
            })

    return sort_characters(chars)


def characters_to_text(chars):
    return "".join(class_names[c["class_id"]] for c in chars)


train_dir = find_split_dir(DATASET_DIR, "train")
valid_dir = find_split_dir(DATASET_DIR, "valid")
test_dir = find_split_dir(DATASET_DIR, "test")

if train_dir is None or valid_dir is None:
    raise FileNotFoundError("Dataset harus memiliki split train dan valid/val")

print("train:", train_dir)
print("valid:", valid_dir)
print("test :", test_dir)


## 5. Dataset PyTorch

Kita crop area yang mencakup seluruh bounding box karakter dengan margin relatif terhadap ukuran teks. Ini hanya dilakukan untuk membuat **training sample OCR** dari anotasi dataset.

Setelah itu:
- grayscale,
- resize menjadi `128 x 32`,
- normalisasi pixel ke `0..1`.

Model hanya menerima gambar hasil crop dan target sequence. Validasi dataset dijalankan sebelum DataLoader dibuat.

In [ ]:
CROP_MARGIN_X = 0.25
CROP_MARGIN_Y = 0.50


def crop_character_region(image, chars):
    height, width = image.shape[:2]

    x1 = min(char["x"] - char["w"] / 2 for char in chars)
    y1 = min(char["y"] - char["h"] / 2 for char in chars)
    x2 = max(char["x"] + char["w"] / 2 for char in chars)
    y2 = max(char["y"] + char["h"] / 2 for char in chars)

    box_width = x2 - x1
    box_height = y2 - y1
    x1 = max(0.0, x1 - box_width * CROP_MARGIN_X)
    y1 = max(0.0, y1 - box_height * CROP_MARGIN_Y)
    x2 = min(1.0, x2 + box_width * CROP_MARGIN_X)
    y2 = min(1.0, y2 + box_height * CROP_MARGIN_Y)

    x1_px = max(0, min(width - 1, int(round(x1 * width))))
    y1_px = max(0, min(height - 1, int(round(y1 * height))))
    x2_px = max(x1_px + 1, min(width, int(round(x2 * width))))
    y2_px = max(y1_px + 1, min(height, int(round(y2 * height))))

    crop = image[y1_px:y2_px, x1_px:x2_px]
    if crop.size == 0:
        raise RuntimeError("Crop karakter kosong")

    return crop


def preprocess_crop(image):
    if image is None or image.size == 0:
        raise ValueError("Gambar crop kosong")

    if image.ndim == 2:
        gray = image
    elif image.ndim == 3 and image.shape[2] == 3:
        # Input array dari OpenCV/YOLO harus BGR.
        gray = cv2.cvtColor(image, cv2.COLOR_BGR2GRAY)
    else:
        raise ValueError(f"Format gambar tidak didukung: {image.shape}")

    gray = cv2.resize(gray, (IMG_W, IMG_H), interpolation=cv2.INTER_AREA)
    return torch.from_numpy(gray).float().div(255.0).unsqueeze(0)


class PlateOCRDataset(Dataset):
    def __init__(self, split_dir):
        if split_dir is None:
            raise ValueError("Split dataset tidak ditemukan")

        self.split_dir = Path(split_dir)
        self.image_dir = self.split_dir / "images"
        self.label_dir = self.split_dir / "labels"

        if not self.image_dir.is_dir() or not self.label_dir.is_dir():
            raise FileNotFoundError(f"Struktur split tidak valid: {self.split_dir}")

        exts = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}
        self.items = []

        for image_path in sorted(self.image_dir.iterdir()):
            if image_path.suffix.lower() not in exts:
                continue

            label_path = self.label_dir / f"{image_path.stem}.txt"
            if label_path.exists() and label_path.stat().st_size > 0:
                self.items.append((image_path, label_path))

    def __len__(self):
        return len(self.items)

    def __getitem__(self, idx):
        image_path, label_path = self.items[idx]
        image = cv2.imread(str(image_path))
        if image is None:
            raise RuntimeError(f"Gagal membaca: {image_path}")

        chars = read_yolo_characters(label_path)
        if not chars:
            raise RuntimeError(f"Tidak ada label karakter: {label_path}")

        crop = preprocess_crop(crop_character_region(image, chars))
        target = torch.tensor(
            [char["class_id"] + 1 for char in chars],
            dtype=torch.long
        )
        text = characters_to_text(chars)

        return crop, target, text


def ctc_required_length(target):
    ids = target.tolist()
    repeated_adjacent = sum(a == b for a, b in zip(ids, ids[1:]))
    return len(ids) + repeated_adjacent


def validate_dataset(dataset, split_name):
    if len(dataset) == 0:
        raise ValueError(f"Split {split_name} tidak memiliki sample berlabel")

    target_lengths = []
    for index in range(len(dataset)):
        _, target, text = dataset[index]
        required_length = ctc_required_length(target)
        if required_length > CTC_TIME_STEPS:
            path = dataset.items[index][0]
            raise ValueError(
                f"Target terlalu panjang untuk CTC di {path}: "
                f"butuh {required_length}, tersedia {CTC_TIME_STEPS}"
            )
        if not text:
            raise ValueError(f"Teks target kosong: {dataset.items[index][0]}")
        target_lengths.append(len(target))

    print(
        f"{split_name}: {len(dataset)} samples | "
        f"target min/max={min(target_lengths)}/{max(target_lengths)}"
    )


train_dataset = PlateOCRDataset(train_dir)
valid_dataset = PlateOCRDataset(valid_dir)
test_dataset = PlateOCRDataset(test_dir) if test_dir is not None else None

validate_dataset(train_dataset, "train")
validate_dataset(valid_dataset, "valid")
if test_dataset is not None:
    validate_dataset(test_dataset, "test")
else:
    print("test: tidak tersedia")


## 6. Cek sample sebelum training

Bagian ini penting. Pastikan teks hasil penyusunan label sesuai urutan karakter pada gambar.

Kalau urutan label salah, model juga akan belajar target yang salah.

In [ ]:
import matplotlib.pyplot as plt

rng = np.random.default_rng(42)
sample_indices = rng.choice(
    len(train_dataset),
    size=min(12, len(train_dataset)),
    replace=False,
)

for i in sample_indices:
    image, target, text = train_dataset[int(i)]

    plt.figure(figsize=(7, 2))
    plt.imshow(image.squeeze(0), cmap="gray")
    plt.title(f"Target: {text}")
    plt.axis("off")
    plt.show()


## 7. DataLoader

CTC menerima target dengan panjang berbeda-beda. Karena itu target dalam satu batch kita gabungkan menjadi satu tensor panjang, lalu menyimpan panjang masing-masing target.

In [ ]:
def collate_fn(batch):
    images, targets, texts = zip(*batch)

    images = torch.stack(images)

    target_lengths = torch.tensor(
        [len(t) for t in targets],
        dtype=torch.long
    )

    targets = torch.cat(targets)

    return images, targets, target_lengths, texts


train_loader = DataLoader(
    train_dataset,
    batch_size=16,
    shuffle=True,
    collate_fn=collate_fn
)

valid_loader = DataLoader(
    valid_dataset,
    batch_size=16,
    shuffle=False,
    collate_fn=collate_fn
)

test_loader = None
if test_dataset is not None:
    test_loader = DataLoader(
        test_dataset,
        batch_size=16,
        shuffle=False,
        collate_fn=collate_fn
    )

images, targets, target_lengths, texts = next(iter(train_loader))

print("Image batch:", images.shape)
print("Flatten targets:", targets.shape)
print("Target lengths:", target_lengths)
print("Text example:", texts[:3])


## 8. Model OCR: CNN -> BiLSTM -> Linear

### CNN
CNN mengubah pixel menjadi feature map visual.

Input:

`[B, 1, 32, 128]`

Setelah CNN kira-kira menjadi:

`[B, 128, 4, 32]`

### Feature map -> sequence
Lebar (`32`) dianggap sebagai **timestep** dari kiri ke kanan.

Menjadi:

`[B, 32, 128*4]`

### BiLSTM
BiLSTM membaca sequence tersebut dan memahami hubungan antar bagian karakter.

### Linear
Setiap timestep menghasilkan score untuk seluruh kemungkinan karakter + `blank`.

In [ ]:
import torch.nn as nn


class CRNN(nn.Module):
    def __init__(self, num_classes):
        super().__init__()

        self.cnn = nn.Sequential(
            nn.Conv2d(1, 32, 3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2, 2),       # 32x128 -> 16x64

            nn.Conv2d(32, 64, 3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2, 2),       # 16x64 -> 8x32

            nn.Conv2d(64, 128, 3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d((2, 1)),     # 8x32 -> 4x32
        )

        self.rnn = nn.LSTM(
            input_size=128 * 4,
            hidden_size=128,
            num_layers=1,
            bidirectional=True,
            batch_first=True
        )

        self.fc = nn.Linear(128 * 2, num_classes)

    def forward(self, x):
        # [B, 1, H, W]
        x = self.cnn(x)

        # [B, C, H, W]
        b, c, h, w = x.shape

        # width dijadikan timestep
        x = x.permute(0, 3, 1, 2)

        # [B, W, C*H]
        x = x.reshape(b, w, c * h)

        # [B, T, 256]
        x, _ = self.rnn(x)

        # [B, T, num_classes]
        x = self.fc(x)

        return x


device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model = CRNN(NUM_CLASSES).to(device)

dummy = torch.randn(2, 1, IMG_H, IMG_W).to(device)
output = model(dummy)

print("Device:", device)
print("Output:", output.shape)
print("Format: [batch, timestep, class]")


## 9. CTC Loss

CTC memungkinkan kita memberi label:

`image -> B1234ABC`

tanpa harus menentukan secara manual:
- pixel mana = `B`,
- timestep mana = `1`,
- timestep mana = `2`, dan seterusnya.

Model bisa menghasilkan sequence seperti:

`blank B B blank 1 1 2 blank 3 4 A A B C`

CTC akan belajar alignment antara timestep model dan sequence target.

In [ ]:
ctc_loss = nn.CTCLoss(
    blank=BLANK_ID,
    zero_infinity=True
)

optimizer = torch.optim.Adam(
    model.parameters(),
    lr=1e-3
)


## 10. Fungsi greedy CTC decoder

Untuk inference sederhana:
1. ambil class dengan score terbesar di setiap timestep,
2. hapus karakter berulang,
3. hapus `blank`,
4. gabungkan menjadi teks.

In [ ]:
def decode_prediction(logits):
    # logits: [B, T, C]
    pred_ids = logits.argmax(dim=2)

    results = []

    for sequence in pred_ids:
        text_tokens = []
        previous = None

        for idx in sequence.tolist():
            if idx != BLANK_ID and idx != previous:
                text_tokens.append(class_names[idx - 1])

            previous = idx

        results.append("".join(text_tokens))

    return results


def decode_targets(flat_targets, lengths):
    results = []
    start = 0

    for length in lengths.tolist():
        ids = flat_targets[start:start + length].tolist()

        text = "".join(
            class_names[idx - 1]
            for idx in ids
        )

        results.append(text)
        start += length

    return results


## 11. Training

Dataset ini kecil, jadi tujuan utama notebook ini adalah memahami proses **CNN + recurrent sequence model + CTC**, bukan mengejar OCR production-grade.

In [ ]:
SEED = 42
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)


def calculate_ctc_loss(logits, targets, target_lengths):
    log_probs = logits.log_softmax(dim=2).permute(1, 0, 2)
    batch_size, time_steps = logits.shape[:2]
    input_lengths = torch.full(
        size=(batch_size,),
        fill_value=time_steps,
        dtype=torch.long,
        device=logits.device
    )
    return ctc_loss(log_probs, targets, input_lengths, target_lengths)


def evaluate_loader(loader):
    model.eval()
    loss_total = 0.0
    correct = 0
    total = 0

    with torch.inference_mode():
        for images, targets, target_lengths, texts in loader:
            images = images.to(device)
            targets = targets.to(device)
            target_lengths = target_lengths.to(device)

            logits = model(images)
            loss_total += calculate_ctc_loss(
                logits, targets, target_lengths
            ).item()

            predictions = decode_prediction(logits.cpu())
            for prediction, truth in zip(predictions, texts):
                correct += int(prediction == truth)
                total += 1

    return loss_total / len(loader), correct / max(total, 1)


EPOCHS = 20
BEST_MODEL_PATH = "plate_crnn_ctc_best.pth"

train_history = []
valid_history = []
best_valid_loss = float("inf")
best_epoch = 0
best_state = None

for epoch in range(EPOCHS):
    model.train()
    train_loss_total = 0.0

    for images, targets, target_lengths, _ in train_loader:
        images = images.to(device)
        targets = targets.to(device)
        target_lengths = target_lengths.to(device)

        optimizer.zero_grad(set_to_none=True)
        logits = model(images)
        loss = calculate_ctc_loss(logits, targets, target_lengths)
        loss.backward()
        optimizer.step()
        train_loss_total += loss.item()

    train_loss = train_loss_total / len(train_loader)
    valid_loss, exact_acc = evaluate_loader(valid_loader)

    train_history.append(train_loss)
    valid_history.append(valid_loss)

    if valid_loss < best_valid_loss:
        best_valid_loss = valid_loss
        best_epoch = epoch + 1
        best_state = {
            key: value.detach().cpu().clone()
            for key, value in model.state_dict().items()
        }
        torch.save({
            "model_state_dict": best_state,
            "class_names": class_names,
            "blank_id": BLANK_ID,
            "img_width": IMG_W,
            "img_height": IMG_H,
            "crop_margin_x": CROP_MARGIN_X,
            "crop_margin_y": CROP_MARGIN_Y,
            "epoch": best_epoch,
            "valid_loss": best_valid_loss,
        }, BEST_MODEL_PATH)

    print(
        f"Epoch {epoch + 1:02d}/{EPOCHS} | "
        f"train_loss={train_loss:.4f} | "
        f"valid_loss={valid_loss:.4f} | "
        f"exact_acc={exact_acc:.3f}"
    )

if best_state is None:
    raise RuntimeError("Tidak ada checkpoint terbaik yang tersimpan")

model.load_state_dict(best_state)
print(f"Best epoch: {best_epoch} | valid_loss={best_valid_loss:.4f}")

if test_loader is not None:
    test_loss, test_exact_acc = evaluate_loader(test_loader)
    print(
        f"test_loss={test_loss:.4f} | "
        f"test_exact_acc={test_exact_acc:.3f}"
    )


## 12. Plot loss

In [ ]:
plt.figure(figsize=(7, 4))
plt.plot(train_history, label="train")
plt.plot(valid_history, label="valid")
plt.xlabel("Epoch")
plt.ylabel("CTC Loss")
plt.legend()
plt.show()


## 13. Lihat hasil prediksi

Perhatikan perbedaan `Ground Truth` dan `Prediction`.

Karena dataset hanya sekitar ratusan gambar, jangan berharap model sederhana ini langsung sangat akurat. Fokus eksperimen adalah melihat bahwa model berhasil belajar memetakan **gambar menjadi sequence karakter**.

In [ ]:
model.eval()

images, targets, target_lengths, texts = next(iter(valid_loader))

with torch.no_grad():
    logits = model(images.to(device))

predictions = decode_prediction(logits.cpu())

for i in range(min(8, len(images))):
    plt.figure(figsize=(7, 2))
    plt.imshow(images[i].squeeze(0), cmap="gray")
    plt.title(
        f"GT: {texts[i]} | Pred: {predictions[i]}"
    )
    plt.axis("off")
    plt.show()


## 14. Simpan model

In [ ]:
MODEL_PATH = BEST_MODEL_PATH

torch.save({
    "model_state_dict": best_state,
    "class_names": class_names,
    "blank_id": BLANK_ID,
    "img_width": IMG_W,
    "img_height": IMG_H,
    "crop_margin_x": CROP_MARGIN_X,
    "crop_margin_y": CROP_MARGIN_Y,
    "best_epoch": best_epoch,
    "valid_loss": best_valid_loss,
}, MODEL_PATH)

print("Saved best model:", MODEL_PATH)


## 15. Inference dari satu gambar crop plat

Fungsi ini menerima **gambar yang sudah berupa crop plat**.

Nantinya hasil YOLO plate detector milikmu cukup di-crop, lalu crop tersebut dikirim ke fungsi ini.

Pipeline final:

`foto kendaraan -> YOLO plate detector -> crop plate -> CRNN OCR -> nomor plat`


In [ ]:
def preprocess_plate(image):
    if isinstance(image, (str, Path)):
        image = cv2.imread(str(image))

    return preprocess_crop(image).unsqueeze(0)


def recognize_plate(plate_image):
    model.eval()

    x = preprocess_plate(plate_image).to(device)

    with torch.inference_mode():
        logits = model(x)

    return decode_prediction(logits.cpu())[0]


# Contoh:
# text = recognize_plate("/content/crop_plate.jpg")
# print("Hasil OCR:", text)


# Yang perlu dipahami dari eksperimen ini

Model **tidak melakukan character detection saat inference**.

Bounding box karakter dari dataset hanya kita gunakan untuk membuat data training `plate crop -> text`.

Saat model sudah dilatih:

`pixels -> CNN -> visual features -> BiLSTM -> sequence features -> Linear -> character scores -> CTC decode -> text`

Itulah konsep dasar CRNN OCR yang sedang dipelajari.

## Catatan
Notebook ini sengaja dibuat sederhana:
- fixed image size `128 x 32`,
- greedy CTC decoding,
- satu baris teks,
- tanpa augmentation,
- tanpa language model,
- tanpa OCR library pretrained.

Setelah versi ini berhasil, peningkatan berikutnya bisa berupa augmentation, preserve aspect ratio, CER (Character Error Rate), beam-search CTC, dan integrasi langsung dengan YOLO plate detector.
